# Day 09. Exercise 02
# Metrics

## 0. Imports

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, roc_auc_score

## 1. Preprocessing

1. Create the same dataframe as in the previous exercise.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

In [3]:
df = pd.read_csv('../data/day-of-week-not-scaled.csv')
df_scaled = pd.read_csv('../data/dayofweek.csv')
df['dayofweek'] = df_scaled['dayofweek']

df.head()

,numTrials,hour,uid_user_0,uid_user_1,uid_user_10,uid_user_11,uid_user_12,uid_user_13,uid_user_14,uid_user_15,...,labname_lab03,labname_lab03s,labname_lab05s,labname_laba04,labname_laba04s,labname_laba05,labname_laba06,labname_laba06s,labname_project1,dayofweek
0,1,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4
1,2,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4
2,3,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4
3,4,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4
4,5,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4


In [4]:
y = df.dayofweek
X = df.drop('dayofweek', axis=1)


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=21, stratify=y)

## 2. SVM

1. Use the best parameters from the previous exercise and train the model of SVM.
2. You need to calculate `accuracy`, `precision`, `recall`, `ROC AUC`.

 - `precision` and `recall` should be calculated for each class (use `average='weighted'`)
 - `ROC AUC` should be calculated for each class against any other class (all possible pairwise combinations) and then weighted average should be applied for the final metric
 - the code in the cell should display the result as below:

```
accuracy is 0.88757
precision is 0.89267
recall is 0.88757
roc_auc is 0.97878
```

In [5]:
def print_metrics(model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)

    print(f'accuracy is {model.score(X_test, y_test):.5f}')
    print(f'precision is {precision_score(y_test, y_pred, average="weighted"):.5f}')
    print(f'recall is {recall_score(y_test, y_pred, average="weighted"):.5f}')
    print(f'roc_auc is {roc_auc_score(y_test, y_proba, multi_class="ovo", average="weighted"):.5f}')

In [6]:
model = SVC(C=10, gamma='auto', random_state=21, probability=True)

print_metrics(model, X_train, y_train, X_test, y_test)

accuracy is 0.88757
precision is 0.89267
recall is 0.88757
roc_auc is 0.97878


## 3. Decision tree

1. The same task for decision tree

In [7]:
model = DecisionTreeClassifier(class_weight='balanced', criterion='gini', max_depth=21, random_state=21)

print_metrics(model, X_train, y_train, X_test, y_test)

accuracy is 0.88462
precision is 0.88765
recall is 0.88462
roc_auc is 0.93528


## 4. Random forest

1. The same task for random forest.

In [8]:
model = RandomForestClassifier(class_weight='balanced', criterion='entropy', max_depth=24, n_estimators=100, random_state=21)

print_metrics(model, X_train, y_train, X_test, y_test)

accuracy is 0.92604
precision is 0.92754
recall is 0.92604
roc_auc is 0.98939


## 5. Predictions

1. Choose the best model.
2. Analyze: for which `weekday` your model makes the most errors (in % of the total number of samples of that class in your full dataset), for which `labname` and for which `users`.
3. Save the model.

In [9]:
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

In [10]:
errors_df = pd.DataFrame({
    'actual': y_test
})

ohe_labname_cols = [c for c in X_test.columns if c.startswith('labname')]
ohe_uid_cols = [c for c in X_test.columns if c.startswith('uid')]

errors_df['labname'] = (
    X_test[ohe_labname_cols]
    .idxmax(axis=1)
    .str.replace('labname_', '')
)

errors_df['uid'] = (
    X_test[ohe_uid_cols]
    .idxmax(axis=1)
    .str.replace('uid_', '')
)


errors_df['is_error'] = y_test != y_pred

errors_df

,actual,labname,uid,is_error
1087,1,project1,user_14,False
16,5,laba04s,user_2,False
563,6,laba05,user_12,False
1381,3,project1,user_20,False
1199,2,project1,user_28,False
...,...,...,...,...
1411,3,project1,user_4,False
1079,1,project1,user_14,False
1222,2,project1,user_29,False
1064,1,project1,user_14,False


In [11]:
weekday_errors = errors_df.groupby('actual').is_error.agg(
    count='size',
    errors_count='sum'
)

weekday_errors['error %'] = np.round(weekday_errors['errors_count'] / weekday_errors['count'] * 100, 2)

weekday_errors.sort_values('error %', ascending=False)

,count,errors_count,error %
actual,,,
0,27,6,22.22
4,21,3,14.29
5,54,5,9.26
1,55,4,7.27
2,30,2,6.67
3,80,3,3.75
6,71,2,2.82


In [12]:
lab_errors = errors_df.groupby('labname').is_error.agg(
    count='size',
    errors_count='sum'
)

lab_errors['error %'] = np.round(lab_errors['errors_count'] / lab_errors['count'] * 100, 2)

lab_errors.sort_values('error %', ascending=False)

,count,errors_count,error %
labname,,,
lab03,1,1,100.00
laba04,35,6,17.14
lab05s,6,1,16.67
laba06s,15,2,13.33
laba06,9,1,11.11
laba04s,25,2,8.00
code_rvw,13,1,7.69
project1,186,10,5.38
laba05,47,1,2.13


In [13]:
uid_errors = errors_df.groupby('uid').is_error.agg(
    count='size',
    errors_count='sum'
)

uid_errors['error %'] = np.round(uid_errors['errors_count'] / uid_errors['count'] * 100, 2)

uid_errors.sort_values('error %', ascending=False).head()

,count,errors_count,error %
uid,,,
user_22,1,1,100.00
user_6,4,2,50.00
user_16,5,1,20.00
user_27,6,1,16.67
user_18,6,1,16.67


## 6. Function

1. Write a function that takes a list of different models and a corresponding list of parameters (dicts) and returns a dict that contains all the 4 metrics for each model.

In [14]:
def calculate_metrics(models, params):
    """
    models - list of models [SVM, DecisionTreeClassifier, RandomForestClassifier]
    params - list of param dicts [{'C': 10, ....}, {}, {}]
    """
    result = {}
    for i, model_class in enumerate(models):
        model = model_class(**params[i])
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)

        accuracy = model.score(X_test, y_test)
        precision = precision_score(y_test, y_pred, average="weighted")
        recall = recall_score(y_test, y_pred, average="weighted")
        roc_auc = roc_auc_score(y_test, y_proba, multi_class="ovo", average="weighted")

        result[model_class.__name__] = [accuracy, precision, recall, roc_auc]

    return result

In [15]:
models = [SVC, DecisionTreeClassifier, RandomForestClassifier]
params = [
    {'C': 10, 'gamma':'auto', 'random_state':21, 'probability':True}, 
    {'class_weight':'balanced', 'criterion':'gini', 'max_depth':21, 'random_state':21}, 
    {'class_weight':'balanced', 'criterion':'entropy', 'max_depth':24, 'n_estimators':100, 'random_state':21}
]

result = calculate_metrics(models, params)

In [16]:
for k, v in result.items():
    print(f'{k} metrics:\n\taccuracy is {v[0]:.5f}\n\tprecision is {v[1]:.5f}\n\trecall is {v[2]:.5f}\n\troc_auc is {v[3]:.5f}\n')

SVC metrics:
	accuracy is 0.88757
	precision is 0.89267
	recall is 0.88757
	roc_auc is 0.97878

DecisionTreeClassifier metrics:
	accuracy is 0.88462
	precision is 0.88765
	recall is 0.88462
	roc_auc is 0.93528

RandomForestClassifier metrics:
	accuracy is 0.92604
	precision is 0.92754
	recall is 0.92604
	roc_auc is 0.98939

